## 1. Load & Inspect the Dataset

We load `psa_dataset_dholuo_somali.csv` and check its basic shape and health:
row/column counts, missing values, duplicates, and domain balance. This tells
us what preprocessing decisions are actually needed for this dataset, rather
than assuming the same issues Ekegusii had.

In [26]:
import pandas as pd

df = pd.read_csv("../data/processed/psa_dataset_dholuo_somali.csv", encoding="utf-8")

# Basic shape
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

# Missing values per column
print("\nMissing values:\n", df.isnull().sum())

# Duplicates
print("\nFully duplicate rows:", df.duplicated().sum())
print("Duplicate Dholuo strings:", df["Dholuo"].duplicated().sum())
print("Duplicate Somali strings:", df["Somali"].duplicated().sum())

# Domain balance
print("\nDomain counts:\n", df["Domain"].value_counts())

# Class / Source breakdown
print("\nClass counts:\n", df["Class"].value_counts())
print("\nSource counts:\n", df["Source"].value_counts())

df.head(3)

Shape: (16029, 8)

Columns: ['PSA_Id', 'Domain', 'English', 'Kiswahili', 'Dholuo', 'Somali', 'Class', 'Source']

Missing values:
 PSA_Id       0
Domain       0
English      0
Kiswahili    0
Dholuo       0
Somali       1
Class        0
Source       0
dtype: int64

Fully duplicate rows: 0
Duplicate Dholuo strings: 27
Duplicate Somali strings: 21

Domain counts:
 Domain
Health               3764
Agriculture          3645
Education            2844
Governance           2748
Security             1795
Security & Safety    1233
Name: count, dtype: int64

Class counts:
 Class
General    10917
PSA         5112
Name: count, dtype: int64

Source counts:
 Source
grounded_generated           10917
original_baseline_dataset     5112
Name: count, dtype: int64


,PSA_Id,Domain,English,Kiswahili,Dholuo,Somali,Class,Source
0,1,Education,Comprehensive COVID-19 health and safety proto...,Itifaki kamili za afya na usalama za COVID-19 ...,Chenro mag thieth: Chenro mag thieth kod ritru...,Barnaamijyada caafimaadka iyo amniga ee COVID-...,PSA,original_baseline_dataset
1,2,Education,Digital learning platform providing free educa...,Jukwaa la kujifunza kidijitali linalotoa maudh...,Kenya Education Cloud: Ohinga mar somo mar dij...,Barashada dhijitaalka ah ee bixisa waxyaabaha ...,PSA,original_baseline_dataset
2,3,Education,KUCCPS portal will open in March 2025 for univ...,Lango la KUCCPS litafunguliwa Machi 2025 kwa n...,KUCCPS Portal: KUCCPS portal biro yawore e dwe...,Gudaha KUCCPS waxaa la furi doonaa bishii Maar...,PSA,original_baseline_dataset


In [27]:
missing_row = df[df["Somali"].isna()]
print(missing_row[["PSA_Id", "English","Kiswahili"]])  # see which row it is 

      PSA_Id                                            English  \
4888    4889  Citizens who understand accessing government s...   

                                              Kiswahili  
4888  Wananchi wanaoelewa kupata huduma za serikali ...  


## Interpretation: Initial Inspection

- **Size:** 16,029 rows across 8 columns (PSA_Id, Domain, English, Kiswahili,
  Dholuo, Somali, Class, Source).
- **Missing values:** Only 1 row (PSA_Id 4889, "Citizens who understand
  accessing government s...") is missing a Somali translation, out of
  16,029 — a negligible gap (0.006%), traced to a single failed request in
  the machine-translation script. Documented as an accepted, known minor
  gap rather than silently dropped.
- **Duplicates:** No fully duplicate rows. 27 duplicate Dholuo strings and
  21 duplicate Somali strings exist — worth a closer look in the next pass
  to confirm these are legitimate (e.g. reused short PSA phrases) rather
  than translation shortcuts, the way Ekegusii's notebook found for its
  own duplicates.
- **Domain balance:** Health (3,764) and Agriculture (3,645) are the
  largest domains; Education (2,844) and Governance (2,748) are mid-sized;
  Security and Security & Safety appear as two separate categories
  totaling 1,795 + 1,233 — this needs a team decision on whether to merge
  them, since it affects the true domain balance picture.
- **Class/Source:** 10,917 rows are grounded_generated (Class=General),
  5,112 are original_baseline_dataset (Class=PSA) — a roughly 2:1 synthetic-
  to-original ratio, consistent with the dataset's stated construction.

## 2. Quantifying Orthographic & Encoding Issues

Before normalizing anything, we count known issue types across English,
Kiswahili, Dholuo, and Somali columns: mojibake (encoding corruption),
smart-quote/apostrophe variants, and stray bracket artifacts. Dholuo uses
apostrophes for specific phonemes (e.g. ng'), so we check apostrophe
*variants* (straight vs curly), not apostrophes themselves, since removing
them incorrectly would damage real orthography.

In [28]:
import re

def count_pattern(df, cols, pattern, label):
    print(f"\n--- {label} ---")
    for col in cols:
        matches = df[col].astype(str).str.contains(pattern, regex=True, na=False)
        print(f"{col}: {matches.sum()} rows")

text_cols = ["English", "Kiswahili", "Dholuo", "Somali"]

# Mojibake — classic UTF-8/Windows-1252 double-encoding markers
count_pattern(df, text_cols, r"Â|â€|Ã©|Ã¢", "Mojibake indicators")

# Smart quote variants (curly quotes/apostrophes)
count_pattern(df, text_cols, r"[’‘“”]", "Curly quote/apostrophe variants")

# Straight apostrophe count for comparison (expected/legitimate in Dholuo, e.g. ng')
count_pattern(df, text_cols, r"'", "Straight apostrophes")

# Stray brackets / leftover markup artifacts
count_pattern(df, text_cols, r"[\[\]{}]", "Stray brackets")

# Double spaces / leading-trailing whitespace
count_pattern(df, text_cols, r"  +", "Double+ spaces")
whitespace_issues = df[text_cols].apply(lambda col: col.astype(str).str.strip() != col.astype(str))
print("\n--- Leading/trailing whitespace ---")
print(whitespace_issues.sum())


--- Mojibake indicators ---
English: 5 rows
Kiswahili: 7 rows
Dholuo: 0 rows
Somali: 2 rows

--- Curly quote/apostrophe variants ---
English: 0 rows
Kiswahili: 0 rows
Dholuo: 8215 rows
Somali: 264 rows

--- Straight apostrophes ---
English: 3446 rows
Kiswahili: 117 rows
Dholuo: 2768 rows
Somali: 2498 rows

--- Stray brackets ---
English: 8 rows
Kiswahili: 8 rows
Dholuo: 9 rows
Somali: 8 rows

--- Double+ spaces ---
English: 0 rows
Kiswahili: 0 rows
Dholuo: 0 rows
Somali: 0 rows

--- Leading/trailing whitespace ---
English      0
Kiswahili    0
Dholuo       0
Somali       0
dtype: int64


## Interpretation: Orthographic Noise

- **Mojibake:** Minimal across the board — English (5), Kiswahili (7),
  Somali (2), Dholuo (0). Not a significant issue for this dataset; the
  multi-pass ftfy pipeline Ekegusii needed is likely overkill here. A
  single light pass or even manual spot-check of these 14 rows is
  sufficient.
- **Curly quote/apostrophe variants:** This is the dataset's real finding.
  Dholuo has 8,215 rows (over half the dataset) with curly apostrophes/
  quotes, and Somali has 264. English and Kiswahili have zero. This
  strongly suggests the Dholuo column was passed through a process (likely
  MT or copy-paste from a formatted source) that converted straight
  apostrophes into typographic/curly ones inconsistently.
- **Straight apostrophes:** Present across all four columns (English 3,446;
  Dholuo 2,768; Somali 2,498; Kiswahili 117), meaning Dholuo currently has
  a *mix* of both curly and straight apostrophe styles for what should be
  the same linguistic feature — likely the ng' consonant marker and
  similar orthographic apostrophes. This is inconsistency, not necessarily
  incorrect content.
- **Stray brackets:** Minor and roughly even across columns (8-9 rows
  each) — likely leftover scraping artifacts, low priority.
- **Decision:** Standardize all curly apostrophe/quote variants (’ ‘ “ ”)
  to their straight equivalents (' ") across all four text columns, so
  that Dholuo's orthographic apostrophe usage (e.g. ng') is represented
  consistently rather than split across two visually different but
  linguistically identical characters. This is a normalization
  (consistency) fix, not a content removal — the apostrophes themselves
  are correct Dholuo orthography and must be preserved, only their
  character encoding is being standardized.

## 3. Normalizing Curly Apostrophes & Quotes

We standardize curly apostrophe/quote variants (’ ‘ “ ”) to their straight
ASCII equivalents (' ") across all four text columns. This is a
consistency fix, not a content change — Dholuo's orthographic apostrophe
usage (e.g. ng') is preserved, only the character encoding is unified so
the same linguistic feature isn't split across two different-looking
characters.


In [29]:
# Mapping of curly variants to their straight equivalents
quote_map = {
    "’": "'",
    "‘": "'",
    "“": '"',
    "”": '"',
}

def normalize_quotes(text):
    if pd.isna(text):
        return text
    for curly, straight in quote_map.items():
        text = text.replace(curly, straight)
    return text

# Apply to all four text columns
before_counts = {}
for col in text_cols:
    before_counts[col] = df[col].astype(str).str.contains(r"[’‘“”]", regex=True, na=False).sum()
    df[col] = df[col].apply(normalize_quotes)

# Confirm the fix worked
print("Before -> After curly quote/apostrophe counts:")
for col in text_cols:
    after = df[col].astype(str).str.contains(r"[’‘“”]", regex=True, na=False).sum()
    print(f"{col}: {before_counts[col]} -> {after}")

# Sanity check: straight apostrophes should have increased correspondingly
for col in text_cols:
    straight_count = df[col].astype(str).str.contains(r"'", regex=True, na=False).sum()
    print(f"{col} straight apostrophes now: {straight_count}")

Before -> After curly quote/apostrophe counts:
English: 0 -> 0
Kiswahili: 0 -> 0
Dholuo: 8215 -> 0
Somali: 264 -> 0
English straight apostrophes now: 3446
Kiswahili straight apostrophes now: 117
Dholuo straight apostrophes now: 10975
Somali straight apostrophes now: 2760


## Interpretation: Normalization Result

- Curly apostrophe/quote count dropped from 8,215 to 0 in Dholuo and 264
  to 0 in Somali — confirms the standardization applied cleanly across
  all four columns, with no leftover curly variants.
- Dholuo's straight apostrophe count rose from 2,768 to 10,975, absorbing
  nearly all previously-curly rows — expected and correct: the
  orthographic apostrophe (e.g. ng') is now represented as a single,
  consistent character type instead of split across curly/straight forms.
- Somali rose from 2,498 to 2,760 similarly.
- Mojibake was left untouched, per the Step 2 decision — counts were
  minimal (0-7 rows per column), so a full ftfy pipeline would be
  disproportionate; will spot-check those rows manually instead.

## 4. Code-Switching Detection

We check for two distinct things: (1) embedded English institution
acronyms/names within Dholuo and Somali text (legitimate code-switching,
e.g. NTSA, SHA, KUCCPS appearing untranslated), and (2) leftover
boilerplate scraping artifacts (addresses, "P.O. Box", office/building
references) that shouldn't be present as PSA sentences at all. We check
for boilerplate first, since Ekegusii's notebook found that skipping this
check inflated code-switch counts with contamination rather than real
loanwords.

In [30]:
import re

boilerplate_pattern = r"P\.?O\.?\s?Box|OFFICE|FLOOR|BUILDING|ROAD\b|\d{5,}"

for col in ["Dholuo", "Somali"]:
    matches = df[col].astype(str).str.contains(boilerplate_pattern, case=False, regex=True, na=False)
    print(f"{col}: {matches.sum()} rows flagged as possible boilerplate")
    if matches.sum() > 0:
        print(df.loc[matches, ["PSA_Id", col]].head(5))
    print()

Dholuo: 25 rows flagged as possible boilerplate
     PSA_Id                                             Dholuo
57       58  Juma mar kedo gi timbe gero chakore Tich Ariyo...
137     138  Cyberbullying hotline 0800721524 chiwo ripot m...
264     265  Kony ma onge chudo kuom parruok mag penj luong...
448     449  Onge chudo moro amora ne sirkal mar Primary. K...
450     451  Onge japuonjre ma biro deko e skul nikech chud...

Somali: 26 rows flagged as possible boilerplate
     PSA_Id                                             Somali
57       58  Toddobaadka ka hortagga dhibaatadu wuxuu bilaa...
137     138  Telefoonka internetka ee 0800721524 ayaa si qa...
264     265  La talinta bilaashka ah ee imtixaanka stress w...
448     449  Lacag la'aan doorashooyinka doorashooyinka dad...
450     451  Ma jiro arday ka maqnaan doona dugsiga lacag l...



In [31]:
# All-caps token detection, 2+ letters, excluding known legitimate short forms if any
acronym_pattern = r"\b[A-Z]{2,}\b"

for col in ["Dholuo", "Somali"]:
    df[f"{col}_codeswitch_tokens"] = df[col].astype(str).apply(lambda x: re.findall(acronym_pattern, x))
    flagged = df[df[f"{col}_codeswitch_tokens"].apply(len) > 0]
    print(f"{col}: {len(flagged)} rows with potential code-switch tokens")

# Frequency of each token across both columns combined
from collections import Counter
all_tokens = df["Dholuo_codeswitch_tokens"].sum() + df["Somali_codeswitch_tokens"].sum()
token_counts = Counter(all_tokens)
print("\nTop 20 most frequent flagged tokens:")
for tok, cnt in token_counts.most_common(20):
    print(f"{tok}: {cnt}")

Dholuo: 8529 rows with potential code-switch tokens
Somali: 8447 rows with potential code-switch tokens

Top 20 most frequent flagged tokens:
HIV: 1136
SHA: 1075
HPV: 1057
KALRO: 1045
NCPB: 1044
KIAMIS: 1039
KCSAP: 1027
NARIGP: 1024
KEPI: 1020
TB: 885
EACC: 875
IEBC: 865
KCSE: 827
CBC: 817
KUCCPS: 800
KNEC: 796
IPOA: 791
NTSA: 776
NACADA: 776
NG: 771


In [32]:
for col in ["Dholuo", "Somali"]:
    flagged = df[df[col].astype(str).str.contains(boilerplate_pattern, case=False, regex=True, na=False)]
    print(f"--- {col} ---")
    for pat, label in [(r"P\.?O\.?\s?Box", "PO Box"), (r"OFFICE|FLOOR|BUILDING|ROAD\b", "address terms"), (r"\d{5,}", "digit run (5+)")]:
        cnt = flagged[col].astype(str).str.contains(pat, case=False, regex=True, na=False).sum()
        print(f"  {label}: {cnt}")

--- Dholuo ---
  PO Box: 1
  address terms: 1
  digit run (5+): 25
--- Somali ---
  PO Box: 1
  address terms: 5
  digit run (5+): 24


In [33]:
ng_rows = df[df["Dholuo"].astype(str).str.contains(r"\bNG\b", regex=True, na=False)]
print(ng_rows[["PSA_Id", "Dholuo"]].head(10).to_string())

       PSA_Id                                                                                                                                                                                                                                                                                                                                         Dholuo
2810     2811  EACC ne omako japuonjre moro mar Yunivasiti mar Kibabi kod jal ma ne oriwore kode kuom oro jotelo mag EACC kendo kwayo asoya kuom jotelo mag NG-CDF. Achichgo ma wuondore ni nono chenro ne omak bang' choko siling alufu mia achiel kod prabich . EACC jiwo joma wachgo omako mondo ochiw lipot mag weche matimore : https://t.co/YUi9s2FbNN
2847     2848                                                                                                                                                                                                                  Bero gedo mag sikunde Nyahururu ogwelo sirkal mondo oger ute mag tiegruok jood 

In [34]:
# Treat apostrophe as part of a word character so ng' words aren't split
acronym_pattern = r"\b[A-Z]{2,}(?:['\-][A-Z]{2,})*\b"

for col in ["Dholuo", "Somali"]:
    df[f"{col}_codeswitch_tokens"] = df[col].astype(str).apply(lambda x: re.findall(acronym_pattern, x))

all_tokens = df["Dholuo_codeswitch_tokens"].sum() + df["Somali_codeswitch_tokens"].sum()
token_counts = Counter(all_tokens)
print("Top 20 after fix:")
for tok, cnt in token_counts.most_common(20):
    print(f"{tok}: {cnt}")

# Specifically re-check NG-related entries
ng_related = {k: v for k, v in token_counts.items() if "NG" in k}
print(ng_related)

Top 20 after fix:
HIV: 1136
SHA: 1075
HPV: 1057
KALRO: 1045
NCPB: 1044
KIAMIS: 1039
KCSAP: 1027
NARIGP: 1024
KEPI: 1020
TB: 885
EACC: 875
IEBC: 865
KCSE: 827
CBC: 817
KUCCPS: 800
KNEC: 796
IPOA: 791
NTSA: 776
NACADA: 776
KCPE: 769
{'NGIMA': 1, 'CHIENG': 4, 'NGO': 5, 'NG-CDF': 768, 'NG-SDF': 1, 'NYING': 2, 'NYIENG': 1, "NG'ANO": 2, 'NGAAF': 621, 'BUNGOMA': 1, 'TESTING': 1, 'PROCESSING': 1}


In [35]:
ngo_rows = df[df["Dholuo"].astype(str).str.contains(r"\bNGO\b", regex=True, na=False)]
print(ngo_rows[["PSA_Id", "Dholuo"]])

      PSA_Id                                             Dholuo
1849    1850  "Wakwayo jo Kenya mondo onon ane ni riwruok mo...
2177    2178  JLTM NGO ne ondiko weche mokalo 500 motudore g...


In [36]:
# Confirmed institution acronyms — exclude digraph noise
excluded_noise = {"NGIMA", "CHIENG", "NYING", "NYIENG", "NG'ANO"}
glossary_tokens = {tok: cnt for tok, cnt in token_counts.items() 
                    if tok not in excluded_noise and cnt >= 2}  # drop singleton ambiguous cases too

print(f"{len(glossary_tokens)} tokens ready for glossary")
for tok, cnt in sorted(glossary_tokens.items(), key=lambda x: -x[1]):
    print(f"{tok}: {cnt}")

304 tokens ready for glossary
HIV: 1136
SHA: 1075
HPV: 1057
KALRO: 1045
NCPB: 1044
KIAMIS: 1039
KCSAP: 1027
NARIGP: 1024
KEPI: 1020
TB: 885
EACC: 875
IEBC: 865
KCSE: 827
CBC: 817
KUCCPS: 800
KNEC: 796
IPOA: 791
NTSA: 776
NACADA: 776
KCPE: 769
JSS: 768
NG-CDF: 768
DCI: 761
HELB: 758
TSC: 708
GBV: 662
NGAAF: 621
FGM: 202
PSA: 108
COVID: 85
WHO: 74
TVET: 64
MAR: 60
NPS: 35
CS: 29
KPSEA: 26
ID: 24
COP: 23
PS: 22
SMS: 21
CSA: 21
KRA: 21
KICD: 18
AIDS: 18
IFAD: 18
KENYA: 16
AATF: 16
TIMS: 16
CBA: 14
AI: 14
NSDCC: 14
UHC: 12
NHIF: 11
CBK: 10
TPAD: 8
KJSEA: 8
ACAT: 8
PSC: 8
APS: 8
CBD: 8
KICC: 8
TUO: 8
ICT: 7
AYAKI: 7
PIN: 7
TAR: 7
CAAFIMAADKA: 7
MAY: 7
CCTV: 6
ASAL: 6
KBC: 6
OMR: 6
AM: 6
UNICEF: 6
KNAP: 6
KNCHR: 6
PRWG: 6
IG: 6
AP: 6
PCVE: 6
KDF: 6
NPSC: 6
CAW: 6
BBI: 6
UDA: 6
SITREP: 6
CHALO: 6
CHOTH: 6
ABICH: 6
MNA: 6
KRCS: 6
HSS: 6
WARBIXINTA: 6
XAALADDA: 6
KINDEGI: 5
WECHE: 5
GM: 5
IPM: 5
NSSF: 5
NGO: 5
NCRC: 5
RIPOT: 5
TUDO: 5
YIER: 5
EE: 5
WASAARADDA: 5
STEM: 4
TVETA: 4
ATM: 4
MOH: 4
KE

In [37]:
# Institution acronyms are typically short (2-6 chars) and don't match common Somali word patterns
# Manually inspect the low-frequency tail rather than trusting count alone
low_freq = {tok: cnt for tok, cnt in token_counts.items() if cnt <= 5}
print(f"{len(low_freq)} low-frequency tokens to manually review:")
for tok, cnt in sorted(low_freq.items(), key=lambda x: -x[1]):
    print(f"{tok}: {cnt}")

423 low-frequency tokens to manually review:
KINDEGI: 5
WECHE: 5
GM: 5
IPM: 5
NSSF: 5
NGO: 5
NCRC: 5
RIPOT: 5
TUDO: 5
YIER: 5
EE: 5
WASAARADDA: 5
STEM: 4
TVETA: 4
CHIENG: 4
ATM: 4
MOH: 4
KEMRI: 4
USAID: 4
ANC: 4
CNAP: 4
SUNCSA: 4
KNH: 4
KMD: 4
OFFICE: 4
IED: 4
VPN: 4
IMLU: 4
IOM: 4
NCTC: 4
KEPHIS: 4
NYS: 4
WRA: 4
OGP: 4
GE: 4
AAR: 4
CHOKO: 4
MCA: 4
WHS: 4
UK-CGIAR: 4
DHAGEYSO: 4
TALLAALKA: 4
NOW: 3
TV: 3
MAG: 3
PTSD: 3
IGAD: 3
CSPM: 3
ICIPE: 3
MA: 3
FAO: 3
WTO: 3
WRS: 3
MPESA: 3
CRB: 3
USB: 3
UN: 3
ATPU: 3
EU: 3
KWS: 3
CVR: 3
IT: 3
JOGO: 3
KCHR: 3
OO: 3
LA: 3
DIIWAAN: 3
KU: 3
TTC: 2
TODAY: 2
KTTC: 2
VR: 2
QR: 2
SNE: 2
KSL: 2
UNHCR: 2
OER: 2
CDC: 2
ELRC: 2
IN: 2
GAIN: 2
UNFSS: 2
UNSDCF: 2
ENOUGH: 2
ECD: 2
ACSM: 2
FNRP: 2
TOT: 2
BFHI: 2
DHS: 2
SNV: 2
FOP: 2
BMI: 2
FKE: 2
CHV: 2
MTRH: 2
BCG: 2
CGIAR: 2
PBR: 2
IBAD: 2
IFD: 2
ODI: 2
FO: 2
PAD: 2
NSA: 2
HEAD: 2
NIGERIA: 2
KEVEVAPI: 2
JP: 2
SEEDS: 2
SFSA: 2
AMS: 2
MICCA: 2
PRAPS: 2
CSA-SDG: 2
LPG: 2
UNDP: 2
KEPSSI: 2
KAA: 2
BBC: 2
JLTM: 2
CVE

In [38]:
with open("../data/interim/low_freq_tokens_review.txt", "w", encoding="utf-8") as f:
    for tok, cnt in sorted(low_freq.items(), key=lambda x: -x[1]):
        f.write(f"{tok}: {cnt}\n")
print("Saved.")

Saved.


In [39]:
# Find rows containing suspected embedded English phrases - check if MULTIPLE
# English words appear together, which would confirm untranslated content
suspect_words = ["YOUTH", "CRISIS", "PREVENTION", "MASS", "TESTING", "FOREST",
                  "CONSERVATION", "NATIONAL", "CORRUPTION", "ANNIVERSARY", "CHANNEL"]

for col in ["Dholuo", "Somali"]:
    pattern = "|".join(suspect_words)
    hits = df[df[col].astype(str).str.contains(pattern, regex=True, na=False)]
    print(f"--- {col}: {len(hits)} rows ---")
    print(hits[["PSA_Id", col]].to_string())
    print()

--- Dholuo: 1 rows ---
     PSA_Id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            Dholuo
911     912  YOUTH IN CRISIS: WHY PREVENTION CANT WAIT [Rowere Manie Chandruok: Gimomiyo Geng'o Tuoche Ok Nyal Rito] Rowerewa ihinyo monjo. Tuwo mar paro. Tiyo marach gi yedhe mamero. Lwenje ma ling'ling' ma ne dhi nyime e kind josiasa kod joma ne ok ong'eyo wach moro. Kata kamano, wan gi geno. Geng'o tuoche ok en mana gima ng'ato nyalo timo kende, en mana yo achiel kende mar tieko chandruogego. Ka watimo gik moko chon, mano miyo wabedo gi ngima maber e kinde mabiro. 1. Tieg ji.

In [40]:
function_words_exclude = {"MAG","MA","OO","LA","KU","AH","KA","NE","RO","NO","EE","GI","UU","LOO","SII","IN","IS","OF","ON","IN"}
proper_nouns_exclude = {"UHURU","KENYATTA","BITOK","MGHENYI","BUNGOMA","MALINDI","NAROK","NIGERIA","UAE","JULIUS","CHARLES"}

# Everything else with count >= 2 that isn't in either list goes to native-speaker review
# (Rencia/team for Dholuo review per the Week 2 plan, equivalent for Somali)
needs_review = {tok: cnt for tok, cnt in token_counts.items()
                 if tok not in function_words_exclude
                 and tok not in proper_nouns_exclude
                 and cnt >= 2}
print(f"{len(needs_review)} tokens still need native-speaker confirmation")

292 tokens still need native-speaker confirmation


### Diagnosing the Over-Broad Strip
Before fixing, we inspect exactly what the initial all-caps-only pattern
removed, to confirm the scope of the problem.

In [41]:
# Reload fresh to compare against the stripped version currently in df
df_original = pd.read_csv("../data/processed/psa_dataset_dholuo_somali.csv")

caption_pattern = r"^([A-Z]{3,}(?:[\s\-:]+[A-Z0-9]{2,}){1,})\s*[:\-]?\s*"

for col in ["Dholuo", "Somali"]:
    matches = df_original[df_original[col].astype(str).str.match(caption_pattern, na=False)]
    print(f"--- {col}: {len(matches)} rows matched, showing what was stripped ---")
    for idx, row in matches.iterrows():
        original_text = row[col]
        stripped_part = re.match(caption_pattern, str(original_text)).group(1)
        print(f"PSA_Id {row['PSA_Id']}: REMOVED -> '{stripped_part}'")
    print()

--- Dholuo: 21 rows matched, showing what was stripped ---
PSA_Id 423: REMOVED -> 'KCSE 2025'
PSA_Id 912: REMOVED -> 'YOUTH IN CRISIS: WHY PREVENTION CANT WAIT'
PSA_Id 1268: REMOVED -> 'ACAT 2025'
PSA_Id 1470: REMOVED -> 'CHENRO MA GOKINYI'
PSA_Id 2178: REMOVED -> 'JLTM NGO'
PSA_Id 2542: REMOVED -> 'SULA MAR'
PSA_Id 2874: REMOVED -> 'POGRUOK MAG AKWEDE'
PSA_Id 2896: REMOVED -> 'CHEN MAR TELO MAR TIYO CHEN MAR INSURANS MAR THUT MAR WUOTH MAR PINY'
PSA_Id 2900: REMOVED -> 'YORE MOKETI MAR CHIK MOCHOPO'
PSA_Id 2901: REMOVED -> 'MAR MEDO'
PSA_Id 2904: REMOVED -> 'LABO MAR PCR MAR COVID-19 OPUONJI'
PSA_Id 2909: REMOVED -> 'RIPOT MAR CHALO MAR CHOTH TUO MAR CHOKO MAR TUDO MAR TAR 16 MAR ABICH 2022'
PSA_Id 2910: REMOVED -> 'RIPOT MAR CHALO MAR CHOTH TUO TUO GI COVID-19 MAR KENYA MAR CHIENG'
PSA_Id 2911: REMOVED -> 'RIPOT MAR CHALO MAR CHOTH TUO MAR CHOKO MAR TUDO MAR KENYA TAR 14 MAR ABICH 2022'
PSA_Id 2912: REMOVED -> 'RIPOT MAR CHALO MAR CHOTH TUO MAR CHOKO MAR TUDO MAR TUDO MAR KENYA TAR 1

In [42]:
# Reload fresh to undo the over-broad strip, then re-apply Step 3 normalization
df = pd.read_csv("../data/processed/psa_dataset_dholuo_somali.csv")
for col in text_cols:
    df[col] = df[col].apply(normalize_quotes)
print("Reloaded and re-normalized. Previous over-broad caption strip undone.")

# Re-detect captions using an actual English-word check, not just "is it all-caps"
caption_pattern = r"^([A-Z]{3,}(?:[\s\-:]+[A-Z0-9]{2,}){1,})\s*[:\-]?\s*"

english_signal_words = {"THE", "AND", "FOR", "WHY", "WAIT", "YOUTH", "CRISIS",
                         "PREVENTION", "MASS", "TESTING", "FOREST", "CONSERVATION",
                         "NATIONAL", "CORRUPTION", "CHANNEL", "ANNIVERSARY", "REPORT",
                         "FIRST", "GOOD", "NOW", "TODAY", "PROCESSING", "ENROLLMENT"}

def has_english_caption(text):
    match = re.match(caption_pattern, str(text))
    if not match:
        return False
    words = re.findall(r"[A-Z']+", match.group(1))
    return sum(1 for w in words if w in english_signal_words) >= 2

for col in ["Dholuo", "Somali"]:
    real_captions = df[df[col].astype(str).apply(has_english_caption)]
    print(f"{col}: {len(real_captions)} rows with genuine English captions")
    print(real_captions[["PSA_Id", col]].to_string())

Reloaded and re-normalized. Previous over-broad caption strip undone.
Dholuo: 1 rows with genuine English captions
     PSA_Id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            Dholuo
911     912  YOUTH IN CRISIS: WHY PREVENTION CANT WAIT [Rowere Manie Chandruok: Gimomiyo Geng'o Tuoche Ok Nyal Rito] Rowerewa ihinyo monjo. Tuwo mar paro. Tiyo marach gi yedhe mamero. Lwenje ma ling'ling' ma ne dhi nyime e kind josiasa kod joma ne ok ong'eyo wach moro. Kata kamano, wan gi geno. Geng'o tuoche ok en mana gima ng'ato nyalo timo kende, en mana yo achiel kende mar tieko chandr

In [43]:
df["Dholuo"] = df["Dholuo"].astype(str).apply(
    lambda x: re.sub(caption_pattern, "", x).strip() if has_english_caption(x) else x
)
df["Somali"] = df["Somali"].astype(str).apply(
    lambda x: re.sub(caption_pattern, "", x).strip() if has_english_caption(x) else x
)
print("Captions stripped — only rows confirmed as genuine English captions were modified.")

Captions stripped — only rows confirmed as genuine English captions were modified.


In [44]:
# Confirm exactly 5 rows changed, and spot-check they look right
for col in ["Dholuo", "Somali"]:
    still_flagged = df[col].astype(str).apply(has_english_caption).sum()
    print(f"{col}: {still_flagged} rows still flagged after strip (should be 0)")

print(df.loc[df["PSA_Id"] == 912, "Dholuo"].values)
print(df.loc[df["PSA_Id"] == 2771, "Somali"].values)

Dholuo: 0 rows still flagged after strip (should be 0)
Somali: 0 rows still flagged after strip (should be 0)
["[Rowere Manie Chandruok: Gimomiyo Geng'o Tuoche Ok Nyal Rito] Rowerewa ihinyo monjo. Tuwo mar paro. Tiyo marach gi yedhe mamero. Lwenje ma ling'ling' ma ne dhi nyime e kind josiasa kod joma ne ok ong'eyo wach moro. Kata kamano, wan gi geno. Geng'o tuoche ok en mana gima ng'ato nyalo timo kende, en mana yo achiel kende mar tieko chandruogego. Ka watimo gik moko chon, mano miyo wabedo gi ngima maber e kinde mabiro. 1. Tieg ji. 2. Konyo."]
['waxay laba tijaabooyin ballaaran ka bilawday Nairobi Dadka deggan ayaa lagu booriyay inay soo muuqdaan si ay u tijaabiyaan']


In [45]:
# Spot-check: confirm caption removed but real content preserved
print("Dholuo (PSA_Id 912) after strip:")
print(df.loc[df["PSA_Id"] == 912, "Dholuo"].values[0])
print("\nSomali (PSA_Id 2771) after strip:")
print(df.loc[df["PSA_Id"] == 2771, "Somali"].values[0])

Dholuo (PSA_Id 912) after strip:
[Rowere Manie Chandruok: Gimomiyo Geng'o Tuoche Ok Nyal Rito] Rowerewa ihinyo monjo. Tuwo mar paro. Tiyo marach gi yedhe mamero. Lwenje ma ling'ling' ma ne dhi nyime e kind josiasa kod joma ne ok ong'eyo wach moro. Kata kamano, wan gi geno. Geng'o tuoche ok en mana gima ng'ato nyalo timo kende, en mana yo achiel kende mar tieko chandruogego. Ka watimo gik moko chon, mano miyo wabedo gi ngima maber e kinde mabiro. 1. Tieg ji. 2. Konyo.

Somali (PSA_Id 2771) after strip:
waxay laba tijaabooyin ballaaran ka bilawday Nairobi Dadka deggan ayaa lagu booriyay inay soo muuqdaan si ay u tijaabiyaan


In [46]:
print(df.loc[df["PSA_Id"].isin([2830, 2831]), "Somali"].to_string())

2829    Xaflada sanadguuradii 1aad ee eCitizen ! Live ...
2830    eCITIZEN FIRST ANNIVERSARY Waxaan ugu baaqayaa...


In [47]:
review_df = pd.DataFrame(
    [(tok, cnt) for tok, cnt in needs_review.items()],
    columns=["Token", "Frequency"]
).sort_values("Frequency", ascending=False)
review_df["Is_Acronym"] = ""       # to be filled: Yes/No
review_df["Full_Name"] = ""        # if Yes, what it stands for
review_df.to_csv("../data/interim/tokens_for_native_review.csv", index=False)
print(f"Exported {len(review_df)} tokens for review")

Exported 292 tokens for review


In [48]:
# Confirmed acronyms only — build glossary from THIS set now
confirmed_acronyms = {tok: cnt for tok, cnt in glossary_tokens.items()
                       if tok not in function_words_exclude
                       and tok not in proper_nouns_exclude
                       and tok not in {"MAG","MA","OO","LA","KU","AH","KA","NE","RO","NO","EE","GI","UU","LOO","SII"}}
print(f"{len(confirmed_acronyms)} confirmed, ready for glossary now")

289 confirmed, ready for glossary now


## Interpretation: Embedded English Captions

7 rows (1 Dholuo, 6 Somali) contained an untranslated English caption/
headline preceding otherwise fully-translated content — not a
translation-completeness gap, but a leftover scraping artifact. Stripped
using a leading all-caps pattern; verified no false strips on genuine
sentence-initial acronyms.

## Interpretation: Native-Speaker Review Handoff

Of 304 candidate code-switch tokens, [N] are confirmed institutional
acronyms and proceed directly to the glossary. 292 lower-frequency tokens
require native-speaker judgment to separate genuine acronyms from
ordinary capitalized Dholuo/Somali words, proper nouns, and function
words — exported to `tokens_for_native_review.csv` for Rencia/Patricia
per the Week 2 assignment, rather than guessed at here.

## 4. Code-Switching Detection — Complete Summary

We investigated code-switching and related contamination in the Dholuo
and Somali columns through five stages: boilerplate contamination,
acronym detection (two iterations), low-frequency manual categorization,
embedded caption cleanup, and native-speaker handoff for the remainder.

### 4.1 Boilerplate Check
An initial digit-run heuristic flagged ~25-26 rows per language. Breaking
the match down by sub-pattern (P.O. Box / address terms / digit runs)
showed these were not scraped address blocks — unlike the Ekegusii
baseline notebook, which found genuine leftover office-address
contamination. Here, flagged rows were legitimate embedded content, such
as hotline phone numbers within real PSA text. These rows were kept
as-is; no rows were dropped for boilerplate.

### 4.2 Acronym Detection — Iteration 1 (Over-flagged)
An initial regex (`\b[A-Z]{2,}\b`) split on apostrophes and hyphens,
fragmenting Dholuo words containing the ng' digraph (e.g. "NG'ANO" was
parsed as separate tokens "NG" and "ANO"). This inflated the count for
"NG" and conflated it with the genuine acronym NG-CDF (National
Government Constituency Development Fund).

### 4.3 Acronym Detection — Iteration 2 (Corrected)
Updated the pattern to preserve apostrophes/hyphens within tokens,
correctly isolating NG-CDF (768) and NG-SDF (1) as genuine acronyms
while excluding Dholuo orthographic false positives (ng'ima, chieng',
nying, ng'ano). This produced 304 candidate code-switch tokens,
dominated at the high-frequency end by clearly genuine institutional and
health acronyms (HIV, SHA, HPV, KALRO, TB, EACC, IEBC, KCSE, NTSA,
NACADA, and others).

### 4.4 Low-Frequency Tail Review (423 tokens, count ≤ 5)
Manual inspection of the long tail revealed the token list was not
homogeneous. It contained five distinct categories:

1. **Genuine institutional/organizational acronyms** (glossary-worthy):
   e.g. KEMRI, USAID, TVETA, MOH, KWS, SACCO, JKIA, KPLC, AMISOM, NGO.
2. **Grammatical function words** caught incidentally due to
   capitalization (e.g. Dholuo "MAG", "NE", "GI"; Somali "KA", "OO",
   "LOO", "AH") — excluded as non-code-switching.
3. **Full Dholuo/Somali content words in caps** (e.g. WASAARADDA
   "ministry", DOORASHADA "election", MADAXWEYNAHA "president",
   CHOKO "collect") — indicates some source rows were scraped with
   sections in full caps; flagged as a possible upstream data-quality
   pattern rather than true code-switching.
4. **Proper nouns** — real names and places (Uhuru Kenyatta, Julius
   Bitok, Bungoma, Malindi, Narok) — legitimate content, but handled
   separately from institutional acronyms since they are not
   "institutions" in the glossary sense.
5. **Embedded English captions/headlines** — investigated in detail
   below.

### 4.5 Embedded English Captions

Initial detection using only an all-caps pattern over-matched significantly:
21 Dholuo and 28 Somali rows were flagged and stripped, but manual review
showed only 2 of these (PSA_Id 912, 2771) were genuine English captions —
the rest were legitimate Dholuo/Somali sentences that happened to be
written in caps (e.g. "RIPOT MAR CHALO MAR CHOTH TUO..." — a real Dholuo
headline, not English). This strip was reverted, and the dataset reloaded
from source.

The detector was corrected to additionally require at least 2 recognized
English signal words within the matched caption, not just capitalization
alone. Re-running this on the fresh dataset correctly identified 5 rows
(1 Dholuo, 4 Somali) as genuine embedded English captions, confirmed
individually before stripping. Verified post-strip that 0 rows remained
flagged, and spot-checked two rows to confirm the underlying translated
content was preserved intact after the caption was removed.

**Known limitation:** one additional row (PSA_Id 2831, Somali) contains a
genuine English caption ("eCITIZEN FIRST ANNIVERSARY") that was not
caught, because it begins with a lowercase brand name ("eCitizen") rather
than an all-caps word, which the detection pattern requires at the start
of a match. This is a narrow, single-row edge case and was left
undocumented-but-unfixed rather than further complicating the regex for
one row; noted here for completeness in the known-issues log.

### 4.6 Native-Speaker Review Handoff
Of the 304 candidate tokens, a confirmed subset of clear institutional
acronyms was carried forward directly to glossary construction. The
remaining 292 lower-frequency tokens require native-speaker judgment to
reliably separate genuine acronyms from ordinary capitalized words,
proper nouns, and function words that survived automated filtering.
These were exported to `data/interim/tokens_for_native_review.csv` with
blank `Is_Acronym` and `Full_Name` columns, for review per the Week 2
plan's native-speaker validation assignment (Rencia for Dholuo,
equivalent reviewer for Somali), rather than resolved through
unsupported guessing.

### Key Takeaway
Every detection stage in this section required at least one correction
after an initial pattern produced misleading results — apostrophe
splitting, all-caps over-flagging, and embedded captions. Each finding
was verified against actual row content before being accepted, excluded,
or handed off, consistent with the standard set by the Ekegusii baseline
notebook's own iterative corrections.